In [18]:
from transformers import  BitsAndBytesConfig
import  torch
from transformers import  AutoModelForCausalLM,AutoTokenizer

In [19]:
quantization_config = BitsAndBytesConfig(
    bnb_in_4bit = True,
    # 使用 NF4 量化
    bnb_4bit_quant_type= 'nf4',
    # 是否使用双重量化
    bnb_4bit_use_double_quant=False,
    # 反量化的数据类型 使用该类型参与前向传播
    bnb_4bit_compute_dtype=torch.bfloat16,
)

In [20]:
# 加载本地路径下的 Qwen-8B 上传至gpu
model = AutoModelForCausalLM.from_pretrained('model/Qwen3-8B',quantization_config=quantization_config).to('cuda')

Loading weights: 100%|██████████| 399/399 [00:46<00:00,  8.62it/s, Materializing param=model.norm.weight]                              


In [21]:
torch.cuda.memory_allocated() / 1024 ** 3

14.280607223510742

In [22]:
from peft import get_peft_model, prepare_model_for_kbit_training
# 冻结参数
prepare_model = prepare_model_for_kbit_training(model)

In [23]:
from peft import LoraConfig
lora_config = LoraConfig(
    r = 4,              # 低秩矩阵的秩
    lora_alpha=4,       # 放缩比例 此处为 4/4 = 1
    lora_dropout=0.05,  # 随机失活 防止过拟合
    bias='none',        # 偏置项
    target_modules=['q_proj','v_proj'], # 目标挂载层
    task_type='CAUSAL_LM'     # 因果语言模型
)
quantized_peft_model = get_peft_model(prepare_model, lora_config)

In [27]:
from datasets import load_dataset

psychology_data = load_dataset("json",data_files={"train":r"./data/psychology_data.jsonl"})
selected_psychology_data = psychology_data['train'].select(range(100))
print(psychology_data)

DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 77250
    })
})


In [25]:
psychology_data["train"].select

<bound method Dataset.select of Dataset({
    features: ['conversation_id', 'category', 'conversation', 'dataset'],
    num_rows: 77250
})>

In [28]:
splited_data =psychology_data['train'].train_test_split(test_size=0.2,train_size=0.8)
splited_data


DatasetDict({
    train: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 61800
    })
    test: Dataset({
        features: ['conversation_id', 'category', 'conversation', 'dataset'],
        num_rows: 15450
    })
})